# 00 — Data Cleaning

Reads every raw CSV from pipeline-a and pipeline-b, applies all cleaning transformations, and saves cleaned versions to `data/cleaned/` subfolders.

**Transformations applied:**
- Column renames (generic names → descriptive; `_code` → `_id` where API `id` field was stored as code; `_title` → `_name`; `amount` → `obligated_amount`; `fiscal_quarter` → `quarter`)
- dtype fixes (budget function/subfunction codes → zero-padded strings; agency id/code float → int)
- Drop fully null columns (`total` in budget quarterly files)
- Drop clutter metadata columns from agency file
- Drop redundant duplicate columns (`award_code`, `recipient_code`)
- Drop null geo rows (no code or name)
- FIPS code formatting for county and district files
- Chunked processing for large files (awards 610 MB, recipients 1.5 GB)

In [1]:
import pandas as pd
from pathlib import Path

ROOT       = Path('/Users/muniprathapmurari/Projects/federal-funding-data-collection')
HIER_RAW   = ROOT / 'pipeline-a-hierarchical' / 'data'
GEO_BASIC  = ROOT / 'pipeline-b-geography'    / 'data' / 'basic-geography'
GEO_AGENCY = ROOT / 'pipeline-b-geography'    / 'data' / 'agency-geography'

HIER_CLEAN       = HIER_RAW   / 'cleaned'
GEO_BASIC_CLEAN  = GEO_BASIC  / 'cleaned'
GEO_AGENCY_CLEAN = GEO_AGENCY / 'cleaned'

for p in [HIER_CLEAN, GEO_BASIC_CLEAN, GEO_AGENCY_CLEAN]:
    p.mkdir(exist_ok=True)

CHUNK = 500_000
print('Output directories ready.')

Output directories ready.


---
## Pipeline A — Hierarchical Data

In [2]:
# ── budget_functions.csv ──────────────────────────────────────────────────────
df = pd.read_csv(HIER_RAW / 'budget_functions.csv')
df = df.rename(columns={
    'budget_function_code':  'budget_function_id',
    'budget_function_title': 'budget_function_name'
})
df['budget_function_id'] = df['budget_function_id'].astype(str).str.zfill(3)
df.to_csv(HIER_CLEAN / 'budget_functions.csv', index=False)
print(f'budget_functions.csv                  {len(df):>6} rows  {df.columns.tolist()}')

# ── budget_subfunctions.csv ───────────────────────────────────────────────────
df = pd.read_csv(HIER_RAW / 'budget_subfunctions.csv')
df = df.rename(columns={
    'budget_subfunction_code':  'budget_subfunction_id',
    'budget_subfunction_title': 'budget_subfunction_name'
})
df['budget_subfunction_id'] = df['budget_subfunction_id'].astype(str).str.zfill(3)
df.to_csv(HIER_CLEAN / 'budget_subfunctions.csv', index=False)
print(f'budget_subfunctions.csv               {len(df):>6} rows  {df.columns.tolist()}')

# ── budget_function_subfunction_mapping.csv ───────────────────────────────────
df = pd.read_csv(HIER_RAW / 'budget_function_subfunction_mapping.csv')
df = df.rename(columns={
    'budget_function_code':     'budget_function_id',
    'budget_function_title':    'budget_function_name',
    'budget_subfunction_code':  'budget_subfunction_id',
    'budget_subfunction_title': 'budget_subfunction_name'
})
df['budget_function_id']    = df['budget_function_id'].astype(str).str.zfill(3)
df['budget_subfunction_id'] = df['budget_subfunction_id'].astype(str).str.zfill(3)
df.to_csv(HIER_CLEAN / 'budget_function_subfunction_mapping.csv', index=False)
print(f'budget_function_subfunction_mapping   {len(df):>6} rows  {df.columns.tolist()}')

budget_functions.csv                      20 rows  ['budget_function_id', 'budget_function_name']
budget_subfunctions.csv                   72 rows  ['budget_subfunction_id', 'budget_subfunction_name']
budget_function_subfunction_mapping       72 rows  ['budget_function_id', 'budget_function_name', 'budget_subfunction_id', 'budget_subfunction_name']


In [3]:
# ── budget_functions_quarterly_all.csv ────────────────────────────────────────
# budget_function_code stored as string in raw file ('050', 'None')
# 'None' string = Unreported Data rows — kept with NaN id
df = pd.read_csv(HIER_RAW / 'budget_functions_quarterly_all.csv', dtype={'budget_function_code': str})
df = df.rename(columns={
    'budget_function_code': 'budget_function_id',
    'amount':               'obligated_amount'
})
df = df.drop(columns=['total'])
df['budget_function_id'] = df['budget_function_id'].replace('None', pd.NA)
df['budget_function_id'] = df['budget_function_id'].apply(
    lambda x: str(int(x)).zfill(3) if pd.notna(x) else pd.NA
)
df.to_csv(HIER_CLEAN / 'budget_functions_quarterly_all.csv', index=False)
print(f'budget_functions_quarterly_all        {len(df):>6} rows  null_ids={df["budget_function_id"].isna().sum()} (Unreported)  {df.columns.tolist()}')

# ── budget_subfunctions_quarterly_all.csv ─────────────────────────────────────
df = pd.read_csv(HIER_RAW / 'budget_subfunctions_quarterly_all.csv')
df = df.rename(columns={
    'budget_function_code':    'budget_function_id',
    'budget_subfunction_code': 'budget_subfunction_id',
    'amount':                  'obligated_amount'
})
df = df.drop(columns=['total'])
df['budget_function_id']    = df['budget_function_id'].astype(str).str.zfill(3)
df['budget_subfunction_id'] = df['budget_subfunction_id'].astype(str).str.zfill(3)
df.to_csv(HIER_CLEAN / 'budget_subfunctions_quarterly_all.csv', index=False)
print(f'budget_subfunctions_quarterly_all     {len(df):>6} rows  {df.columns.tolist()}')

budget_functions_quarterly_all           621 rows  null_ids=32 (Unreported)  ['fy', 'quarter', 'budget_function_id', 'budget_function_name', 'obligated_amount']
budget_subfunctions_quarterly_all       2092 rows  ['fy', 'quarter', 'budget_function_id', 'budget_function_name', 'budget_subfunction_id', 'budget_subfunction_name', 'obligated_amount']


In [4]:
# ── agency_ALL_FY.csv ─────────────────────────────────────────────────────────
# FY2017-2019: quarterly only. FY2020: mixed (Q1-Q2 quarterly, Q3-Q4 period).
# FY2021-2024: period only. Amounts are cumulative YTD in both granularities,
# so convert period rows by keeping only end-of-quarter periods:
# P3→Q1, P6→Q2, P9→Q3, P12→Q4 (the value at end of each quarter IS the quarterly cumulative).
PERIOD_TO_QUARTER = {3: 1, 6: 2, 9: 3, 12: 4}

raw = pd.read_csv(HIER_RAW / 'agency_ALL_FY.csv')

qtrs    = raw[raw['time_granularity'] == 'quarter'].copy()
periods = raw[raw['time_granularity'] == 'period'].copy()
periods = periods[periods['fiscal_period'].isin(PERIOD_TO_QUARTER.keys())].copy()
periods['fiscal_quarter'] = periods['fiscal_period'].map(PERIOD_TO_QUARTER).astype(int)

df = pd.concat([qtrs, periods], ignore_index=True)

# FY2020 overlap: Q1-Q2 exist in both — keep quarterly row over period-derived row
df = df.sort_values('time_granularity').drop_duplicates(
    subset=['fy', 'fiscal_quarter', 'id'], keep='first'
)

df = df.rename(columns={
    'id':             'agency_id',
    'code':           'agency_code',
    'type':           'agency_type',
    'name':           'agency_name',
    'fiscal_quarter': 'quarter',
    'amount':         'obligated_amount'
})
df = df.drop(columns=['time_granularity', 'fiscal_period', 'fyq', 'fyp', 'link'])
before = len(df)
df = df.dropna(subset=['agency_id'])
df['agency_id']   = df['agency_id'].astype(int)
df['agency_code'] = df['agency_code'].astype(int)
df = df.sort_values(['fy', 'quarter', 'agency_id']).reset_index(drop=True)
df.to_csv(HIER_CLEAN / 'agency_ALL_FY.csv', index=False)
print(f'agency_ALL_FY                         {len(df):>6} rows  (dropped {before - len(df)} null rows)')
print(f'  FY range: {df["fy"].min()} – {df["fy"].max()}  quarters: {sorted(df["quarter"].unique())}')
print(f'  columns: {df.columns.tolist()}')

agency_ALL_FY                           3331 rows  (dropped 31 null rows)
  FY range: 2017 – 2024  quarters: [1, 2, 3, 4]
  columns: ['fy', 'quarter', 'agency_id', 'agency_code', 'agency_type', 'agency_name', 'obligated_amount']


In [5]:
# ── federal_accounts_ALL_FY.csv ───────────────────────────────────────────────
# federal_account_code stores API id field (not the real code) → rename to federal_account_id
df = pd.read_csv(HIER_RAW / 'federal_accounts_ALL_FY.csv')
df = df.rename(columns={
    'budget_function_code':    'budget_function_id',
    'budget_subfunction_code': 'budget_subfunction_id',
    'federal_account_code':    'federal_account_id'
})
df['budget_function_id']    = df['budget_function_id'].astype(str).str.zfill(3)
df['budget_subfunction_id'] = df['budget_subfunction_id'].astype(str).str.zfill(3)
df.to_csv(HIER_CLEAN / 'federal_accounts_ALL_FY.csv', index=False)
print(f'federal_accounts_ALL_FY               {len(df):>6} rows  {df.columns.tolist()}')

# ── federal_accounts_with_agency_ALL_FY.csv ───────────────────────────────────
# fiscal_period is 100% null (never populated in collection) → drop
df = pd.read_csv(HIER_RAW / 'federal_accounts_with_agency_ALL_FY.csv')
df = df.drop(columns=['fiscal_period'])
df = df.rename(columns={
    'agency':         'agency_id',
    'id':             'federal_account_id',
    'code':           'federal_account_code',
    'type':           'federal_account_type',
    'name':           'federal_account_name',
    'fiscal_quarter': 'quarter',
    'amount':         'obligated_amount',
    'account_number': 'federal_account_number'
})
df.to_csv(HIER_CLEAN / 'federal_accounts_with_agency_ALL_FY.csv', index=False)
print(f'federal_accounts_with_agency_ALL_FY   {len(df):>6} rows  {df.columns.tolist()}')

federal_accounts_ALL_FY                59084 rows  ['fy', 'quarter', 'budget_function_id', 'budget_subfunction_id', 'federal_account_id', 'federal_account_name', 'obligated_amount']
federal_accounts_with_agency_ALL_FY    58830 rows  ['fy', 'quarter', 'agency_id', 'federal_account_id', 'federal_account_code', 'federal_account_type', 'federal_account_name', 'obligated_amount', 'federal_account_number']


In [6]:
# ── awards_ALL_FY.csv  (610 MB — chunked) ────────────────────────────────────
# Drop award_code: identical to award_name in 99.99% of rows
# award_id float64 → Int64 (nullable int)
out = HIER_CLEAN / 'awards_ALL_FY.csv'
total = 0
for i, chunk in enumerate(pd.read_csv(HIER_RAW / 'awards_ALL_FY.csv', chunksize=CHUNK)):
    chunk = chunk.rename(columns={
        'budget_function_code':    'budget_function_id',
        'budget_subfunction_code': 'budget_subfunction_id',
        'federal_account_code':    'federal_account_id'
    })
    chunk = chunk.drop(columns=['award_code'])
    chunk['award_id'] = chunk['award_id'].astype('Int64')
    chunk.to_csv(out, index=False, mode='w' if i == 0 else 'a', header=(i == 0))
    total += len(chunk)
    print(f'  awards chunk {i+1} done  ({total:,} rows so far)')
print(f'awards_ALL_FY                       {total:>8,} rows  done')

  awards chunk 1 done  (500,000 rows so far)
  awards chunk 2 done  (1,000,000 rows so far)
  awards chunk 3 done  (1,500,000 rows so far)
  awards chunk 4 done  (2,000,000 rows so far)
  awards chunk 5 done  (2,500,000 rows so far)
  awards chunk 6 done  (3,000,000 rows so far)
  awards chunk 7 done  (3,500,000 rows so far)
  awards chunk 8 done  (4,000,000 rows so far)
  awards chunk 9 done  (4,500,000 rows so far)
  awards chunk 10 done  (5,000,000 rows so far)
  awards chunk 11 done  (5,500,000 rows so far)
  awards chunk 12 done  (6,000,000 rows so far)
  awards chunk 13 done  (6,500,000 rows so far)
  awards chunk 14 done  (7,000,000 rows so far)
  awards chunk 15 done  (7,500,000 rows so far)
  awards chunk 16 done  (8,000,000 rows so far)
  awards chunk 17 done  (8,333,844 rows so far)
awards_ALL_FY                       8,333,844 rows  done


In [7]:
# ── recipients_ALL_FY.csv  (1.5 GB — chunked) ────────────────────────────────
# Drop recipient_code: identical to recipient_name in 99.9% of rows
out = HIER_CLEAN / 'recipients_ALL_FY.csv'
total = 0
for i, chunk in enumerate(pd.read_csv(HIER_RAW / 'recipients_ALL_FY.csv', chunksize=CHUNK)):
    chunk = chunk.rename(columns={
        'budget_function_code':    'budget_function_id',
        'budget_subfunction_code': 'budget_subfunction_id',
        'federal_account_code':    'federal_account_id'
    })
    chunk = chunk.drop(columns=['recipient_code'])
    chunk.to_csv(out, index=False, mode='w' if i == 0 else 'a', header=(i == 0))
    total += len(chunk)
    print(f'  recipients chunk {i+1} done  ({total:,} rows so far)')
print(f'recipients_ALL_FY                   {total:>8,} rows  done')

  recipients chunk 1 done  (500,000 rows so far)
  recipients chunk 2 done  (1,000,000 rows so far)
  recipients chunk 3 done  (1,500,000 rows so far)
  recipients chunk 4 done  (2,000,000 rows so far)
  recipients chunk 5 done  (2,500,000 rows so far)
  recipients chunk 6 done  (3,000,000 rows so far)
  recipients chunk 7 done  (3,500,000 rows so far)
  recipients chunk 8 done  (4,000,000 rows so far)
  recipients chunk 9 done  (4,500,000 rows so far)
  recipients chunk 10 done  (5,000,000 rows so far)
  recipients chunk 11 done  (5,500,000 rows so far)
  recipients chunk 12 done  (6,000,000 rows so far)
  recipients chunk 13 done  (6,500,000 rows so far)
  recipients chunk 14 done  (7,000,000 rows so far)
  recipients chunk 15 done  (7,500,000 rows so far)
  recipients chunk 16 done  (8,000,000 rows so far)
  recipients chunk 17 done  (8,500,000 rows so far)
  recipients chunk 18 done  (9,000,000 rows so far)
  recipients chunk 19 done  (9,500,000 rows so far)
  recipients chunk 20 d

---
## Pipeline B — Geography Data

In [8]:
# ── Basic geography files ─────────────────────────────────────────────────────
# All files: rename code→geo_code, name→geo_name, amount→obligated_amount
# Drop geo_layer (constant single value per file, redundant with filename)
# Drop null geo_code rows
# county + district extra: geo_code float → str (FIPS); state_code float → str zfill(2)

def clean_geo_basic(fname, has_state_code=False):
    df = pd.read_csv(GEO_BASIC / fname)
    df = df.rename(columns={'code': 'geo_code', 'name': 'geo_name', 'amount': 'obligated_amount'})
    df = df.drop(columns=['geo_layer'])
    before = len(df)
    df = df.dropna(subset=['geo_code'])
    if has_state_code:
        df['geo_code']   = df['geo_code'].astype(int).astype(str)
        df['state_code'] = df['state_code'].astype(int).astype(str).str.zfill(2)
    df.to_csv(GEO_BASIC_CLEAN / fname, index=False)
    print(f'{fname:<50} {len(df):>7} rows  (dropped {before - len(df)} null rows)  {df.columns.tolist()}')

clean_geo_basic('geography_state_all_FY2008_2024.csv')
clean_geo_basic('geography_country_all_FY2008_2024.csv')
clean_geo_basic('geography_county_all_FY2008_2024.csv',   has_state_code=True)
clean_geo_basic('geography_district_all_FY2008_2024.csv')

geography_state_all_FY2008_2024.csv                   3808 rows  (dropped 68 null rows)  ['geo_code', 'geo_name', 'obligated_amount', 'population', 'fy', 'quarter']
geography_country_all_FY2008_2024.csv                14909 rows  (dropped 68 null rows)  ['geo_code', 'geo_name', 'obligated_amount', 'population', 'fy', 'quarter']
geography_county_all_FY2008_2024.csv                132441 rows  (dropped 41 null rows)  ['geo_code', 'geo_name', 'obligated_amount', 'population', 'fy', 'quarter', 'state_code', 'state_name']
geography_district_all_FY2008_2024.csv                3969 rows  (dropped 9 null rows)  ['geo_code', 'geo_name', 'obligated_amount', 'population', 'fy', 'quarter']


In [9]:
# ── Agency geography files ────────────────────────────────────────────────────
# All files: rename code→geo_code, name→geo_name, amount→obligated_amount
# Drop geo_layer (constant single value per file, redundant with filename)
# Drop null geo_code rows
# county files extra: geo_code float → str; state_code float → str zfill(2)

GEO_AGENCY_FILES = [
    ('geo_state_funding_ALL_FY.csv',     False),
    ('geo_state_awarding_ALL_FY.csv',    False),
    ('geo_country_funding_ALL_FY.csv',   False),
    ('geo_country_awarding_ALL_FY.csv',  False),
    ('geo_county_funding_ALL_FY.csv',    True),
    ('geo_county_awarding_ALL_FY.csv',   True),
    ('geo_district_funding_ALL_FY.csv',  False),
    ('geo_district_awarding_ALL_FY.csv', False),
]

def clean_geo_agency(fname, has_state_code=False):
    df = pd.read_csv(GEO_AGENCY / fname)
    df = df.rename(columns={'code': 'geo_code', 'name': 'geo_name', 'amount': 'obligated_amount'})
    df = df.drop(columns=['geo_layer'])
    before = len(df)
    df = df.dropna(subset=['geo_code'])
    if has_state_code:
        df['geo_code']   = df['geo_code'].astype(int).astype(str)
        df['state_code'] = df['state_code'].astype(int).astype(str).str.zfill(2)
    df.to_csv(GEO_AGENCY_CLEAN / fname, index=False)
    print(f'{fname:<50} {len(df):>7} rows  (dropped {before - len(df)} null rows)')

for fname, has_state in GEO_AGENCY_FILES:
    clean_geo_agency(fname, has_state_code=has_state)

geo_state_funding_ALL_FY.csv                         14878 rows  (dropped 61 null rows)
geo_state_awarding_ALL_FY.csv                        15181 rows  (dropped 0 null rows)
geo_country_funding_ALL_FY.csv                        8659 rows  (dropped 1 null rows)
geo_country_awarding_ALL_FY.csv                       8793 rows  (dropped 19 null rows)
geo_county_funding_ALL_FY.csv                       372028 rows  (dropped 242 null rows)
geo_county_awarding_ALL_FY.csv                      826507 rows  (dropped 335 null rows)
geo_district_funding_ALL_FY.csv                     142133 rows  (dropped 362 null rows)
geo_district_awarding_ALL_FY.csv                    213679 rows  (dropped 611 null rows)


In [10]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=== Cleaned files ===')
for folder in [HIER_CLEAN, GEO_BASIC_CLEAN, GEO_AGENCY_CLEAN]:
    print(f'\n{folder.relative_to(ROOT)}')
    for f in sorted(folder.iterdir()):
        mb = f.stat().st_size / 1_048_576
        print(f'  {f.name:<55} {mb:6.1f} MB')

=== Cleaned files ===

pipeline-a-hierarchical/data/cleaned
  agency_ALL_FY.csv                                          0.2 MB
  awards_ALL_FY.csv                                        474.5 MB
  budget_function_subfunction_mapping.csv                    0.0 MB
  budget_functions.csv                                       0.0 MB
  budget_functions_quarterly_all.csv                         0.0 MB
  budget_subfunctions.csv                                    0.0 MB
  budget_subfunctions_quarterly_all.csv                      0.2 MB
  federal_accounts_ALL_FY.csv                                6.1 MB
  federal_accounts_with_agency_ALL_FY.csv                    7.5 MB
  recipients_ALL_FY.csv                                   1193.7 MB

pipeline-b-geography/data/basic-geography/cleaned
  geography_country_all_FY2008_2024.csv                      0.6 MB
  geography_county_all_FY2008_2024.csv                       7.6 MB
  geography_district_all_FY2008_2024.csv                     0.2 MB
  geo